# AIML Lab Assignment - Practical Assignment 03
## Performance Evaluation of Regression Model

**Topic:** Electric Vehicle Range Prediction

**Problem Statement:**  
Develop a regression model for a real-world application and evaluate its performance using suitable metrics. The Electric Vehicle dataset is used to model real-world driving range from battery capacity. A Linear Regression model is trained, evaluated, and compared with a manually implemented Gradient Descent optimization method.


## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Dataset

In [ ]:
uploaded = files.upload()

file_path = next(iter(uploaded))
df = pd.read_csv(file_path)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())
print("\nColumn Names:")
print(list(df.columns))


## Select Variables
Set the two names below according to the columns shown above.

In [ ]:
# Change only these two names if your CSV uses different column names.
X_column = 'battery_capacity_kWh'
y_column = 'real_world_range_km'

data = df[[X_column, y_column]].copy()

print("Selected input:", X_column)
print("Selected target:", y_column)


## Cleaning

In [ ]:
data[X_column] = pd.to_numeric(data[X_column], errors='coerce')
data[y_column] = pd.to_numeric(data[y_column], errors='coerce')

invalid_battery = int((data[X_column] <= 0).sum())
invalid_range = int((data[y_column] <= 0).sum())

data.loc[data[X_column] <= 0, X_column] = np.nan
data.loc[data[y_column] <= 0, y_column] = np.nan

before = len(data)
data = data.dropna().reset_index(drop=True)

print("Rows before cleaning:", before)
print("Invalid battery capacity values:", invalid_battery)
print("Invalid range values:", invalid_range)
print("Rows removed:", before - len(data))
print("Rows after cleaning:", len(data))
display(data.head(10))


## Split

In [ ]:
X = data[[X_column]]
y = data[y_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


## Linear Regression

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

sk_b = model.intercept_
sk_w = model.coef_[0]
y_pred = model.predict(X_test)

print(f"Intercept: {sk_b:.6f}")
print(f"Slope: {sk_w:.6f}")
print(f"Regression Equation: EV Range = {sk_b:.2f} + ({sk_w:.2f} × Battery Capacity)")


## Gradient Descent

In [ ]:
def gradient_descent(X, y, learning_rate=0.0001, epochs=5000):
    x = np.asarray(X, dtype=float).reshape(-1)
    target = np.asarray(y, dtype=float).reshape(-1)

    valid = np.isfinite(x) & np.isfinite(target)
    x = x[valid]
    target = target[valid]

    n = len(x)
    w = 0.0
    b = 0.0
    history = []

    for _ in range(epochs):
        pred = w * x + b
        error = pred - target

        history.append(np.mean(error ** 2))

        dw = (2 / n) * np.sum(x * error)
        db = (2 / n) * np.sum(error)

        w -= learning_rate * dw
        b -= learning_rate * db

    return w, b, np.array(history)


In [ ]:
gd_w, gd_b, loss_history = gradient_descent(
    X_train[X_column], y_train, 0.0001, 5000
)

gd_pred = gd_w * X_test[X_column].to_numpy() + gd_b

print("Learning rate: 0.0001")
print("Epochs: 5000")
print(f"Intercept: {gd_b:.6f}")
print(f"Slope: {gd_w:.6f}")
print(f"Final training MSE: {loss_history[-1]:.6f}")


## Metrics

In [ ]:
metrics = pd.DataFrame({
    'MAE': [
        mean_absolute_error(y_test, y_pred),
        mean_absolute_error(y_test, gd_pred)
    ],
    'MSE': [
        mean_squared_error(y_test, y_pred),
        mean_squared_error(y_test, gd_pred)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, y_pred)),
        np.sqrt(mean_squared_error(y_test, gd_pred))
    ],
    'R2': [
        r2_score(y_test, y_pred),
        r2_score(y_test, gd_pred)
    ]
}, index=['Linear Regression', 'Gradient Descent'])

display(metrics.round(4))


## Predictions

In [ ]:
predictions = pd.DataFrame({
    'Battery Capacity (kWh)': X_test[X_column].to_numpy(),
    'Actual EV Range (km)': y_test.to_numpy(),
    'Linear Regression': y_pred.round(2),
    'Gradient Descent': gd_pred.round(2)
}).sort_values('Battery Capacity (kWh)')

display(predictions)


## Regression Plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(data[X_column], data[y_column], alpha=0.75, label='EV data')

x_line = np.linspace(data[X_column].min(), data[X_column].max(), 200)
plt.plot(x_line, sk_b + sk_w * x_line, linewidth=2, label='Regression line')

plt.xlabel('Battery Capacity (kWh)')
plt.ylabel('Real-World EV Range (km)')
plt.title('Battery Capacity vs EV Range')
plt.legend()
plt.grid(True)
plt.show()


## Convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(loss_history) + 1), loss_history)

plt.xlabel('Epoch')
plt.ylabel('Training MSE')
plt.title('Gradient Descent Convergence')
plt.grid(True)
plt.show()


## Learning Rate

In [ ]:
learning_rates = [0.00001, 0.00005, 0.0001]
lr_results = []

for rate in learning_rates:
    w, b, history = gradient_descent(
        X_train[X_column], y_train, rate, 5000
    )
    lr_results.append([rate, 5000, w, b, history[-1]])

lr_results = pd.DataFrame(
    lr_results,
    columns=['Learning Rate', 'Epochs', 'Slope', 'Intercept', 'Final Train MSE']
)

display(lr_results.round(6))


## Residuals

In [ ]:
residuals = y_test.to_numpy() - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals)
plt.axhline(0, linestyle='--')

plt.xlabel('Predicted EV Range (km)')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.grid(True)
plt.show()


## New Prediction

In [ ]:
new_data = pd.DataFrame({X_column: [50]})
prediction = model.predict(new_data)[0]

print(f"Predicted EV range for 50 kWh battery: {prediction:.2f} km")
print(
    "Parameters close:",
    np.isclose(sk_w, gd_w, atol=0.001)
    and np.isclose(sk_b, gd_b, atol=0.001)
)


## Comparison

In [ ]:
print("Comparison")
display(metrics.round(4))


## Result

In [ ]:
print(f"Regression equation: EV Range = {sk_b:.2f} + ({sk_w:.2f} × Battery Capacity)")
print(f"Test R2 = {r2_score(y_test, y_pred):.4f}")
print(f"Prediction for 50 kWh battery = {prediction:.2f} km")
